# Trading Intelligent — CH02 v4.0
## Laboratoire 2 — AR, MA, ARMA et ARIMA : comprendre les mécanismes avant d'ajuster

**Auteur académique : Tawfik Masrour**

**Position dans le cours :** Chapitre 2 — *Comprendre la dépendance temporelle : stationnarité, mémoire, ARIMA et saisonnalité*.

### Question directrice
Comment distinguer la mémoire des valeurs de la mémoire des chocs, puis construire un modèle ARIMA sans transformer ACF, PACF ou AIC en recettes automatiques ?

> **Contrat pédagogique du laboratoire.** On ne demande jamais au code de nous dire ce que nous n'avons pas encore défini. Chaque expérience suit le fil : **question → intuition → calcul → graphique → diagnostic → interprétation financière**.

> Ce notebook reprend intégralement les expériences pédagogiques du laboratoire historique correspondant, mais les replace dans le cours **Trading Intelligent** et les enrichit par des explications, contrôles et ponts vers la finance.


### Carte mentale du laboratoire

Nous allons rencontrer quatre familles, mais une seule idée les relie : **où se trouve la mémoire ?**

- **AR** : la valeur passée du système se propage dans le présent ;
- **MA** : les chocs passés continuent d'agir directement ;
- **ARMA** : les deux mémoires coexistent ;
- **ARIMA** : on transforme d'abord une série non stationnaire, puis on modélise la dynamique restante.

Avant toute estimation, nous construirons les mécanismes sur des séries synthétiques dont nous connaissons la vérité. C'est le seul moyen de savoir ensuite ce qu'un diagnostic peut réellement identifier.


##**1️⃣ Setup & Outils d'Analyse**
Nous avons besoin de générer des processus (simulation) et de les analyser (ACF/PACF).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# Configuration Esthétique
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.titlesize'] = 15

print("✅ Atelier de Mécanique prêt.")

##**2️⃣ Partie I : Le Processus Auto-Régressif (AR)**

L'AR est la "***Mémoire du Passé***".

La valeur d'aujourd'hui est une fraction de la valeur d'hier.
$$y_t = \phi_1 y_{t-1} + \epsilon_t$$

Nous allons simuler un AR(1) avec $\phi_1 = 0.8$. Cela signifie une ***forte mémoire (persistance***).

### AR(1) à la main : un choc doit pouvoir être suivi pas à pas

Pour

\[
X_t=\phi X_{t-1}+
arepsilon_t,
\]

imaginons un choc de \(+10\) au temps 0 puis aucun nouveau choc.

Si \(\phi=0.8\),

\[
10 
ightarrow 8 
ightarrow 6.4 
ightarrow 5.12 
ightarrow \cdots
\]

L'effet après \(k\) périodes est \(10\phi^k\). La mémoire n'est donc pas une métaphore : elle est une **suite géométrique**.

- \(0<\phi<1\) : dissipation monotone ;
- \(-1<\phi<0\) : dissipation oscillante ;
- \(\phi=1\) : le choc ne disparaît plus ;
- \(|\phi|>1\) : le choc est amplifié.

Cette lecture précède volontairement ACF/PACF : on comprend d'abord la dynamique qui produira ensuite leur forme.


In [ ]:
# --- SIMULATION AR(1) ---
# Note technique : Statsmodels utilise la convention phi polynomiale : (1 - phi*L)y = epsilon
# Donc pour y_t = 0.8*y_{t-1}, on écrit ar = [1, -0.8]
ar_params = np.array([1, -0.8])
ma_params = np.array([1]) # Pas de MA

# Génération du processus
ar1_process = ArmaProcess(ar_params, ma_params)
np.random.seed(42)
y_ar1 = ar1_process.generate_sample(nsample=300)

# --- VISUALISATION ---
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# 1. Série Temporelle
ax[0].plot(y_ar1, color='#3b82f6')
ax[0].set_title(r"Série AR(1) avec $\phi=0.8$")
ax[0].set_xlabel("Temps")

# 2. ACF (Autocorrélation) - Décroissance Lente
plot_acf(y_ar1, lags=20, ax=ax[1], title="ACF : Décroissance Lente (Signature)")

# 3. PACF (Partielle) - Coupure Nette
plot_pacf(y_ar1, lags=20, ax=ax[2], title="PACF : Coupure à p=1 (Signature)")
plt.show()

##**📝 Analyse de la Signature (AR)**

Regardez bien les graphiques 2 et 3 :
**ACF (Milieu)** : Elle descend doucement en toboggan. C'est l'effet de "ricochet" (hier influence aujourd'hui, qui influence demain...).

**PACF (Droite)** : Elle s'arrête net après le Lag 1.

**Règle d'Or** : ***Si la PACF coupe après le lag $p$, c'est un processus AR($p$)***.

#**3️⃣ Partie II : Le Processus Moyenne Mobile (MA)**

Le ***MA*** est la "***Mémoire du Choc***".

La valeur d'aujourd'hui dépend du bruit d'hier.$$y_t = \epsilon_t + \theta_1 \epsilon_{t-1}$$

Simulons un MA(1) avec $\theta_1 = 0.7$.

### MA n'est pas une moyenne glissante

Le terme *Moving Average* est historiquement trompeur pour un débutant. Dans

\[
X_t=
arepsilon_t+	heta
arepsilon_{t-1},
\]

nous ne faisons pas la moyenne des dernières observations \(X\). Nous combinons des **innovations présentes et passées**.

Un choc \(
arepsilon_t\) touche \(X_t\) immédiatement puis \(X_{t+1}\) via \(	heta\), et disparaît ensuite dans un MA(1). C'est donc une mémoire **directe et finie des chocs**.


In [ ]:
# --- SIMULATION MA(1) ---
# ar = [1] (Pas d'AR)
# ma = [1, 0.7] (y_t = e_t + 0.7*e_{t-1})
ar_params = np.array([1])
ma_params = np.array([1, 0.7])

ma1_process = ArmaProcess(ar_params, ma_params)
np.random.seed(42)
y_ma1 = ma1_process.generate_sample(nsample=300)

# --- VISUALISATION ---
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# 1. Série
ax[0].plot(y_ma1, color='#10b981')
ax[0].set_title(r"Série MA(1) avec $\theta=0.7$")

# 2. ACF - Coupure Nette
plot_acf(y_ma1, lags=20, ax=ax[1], title="ACF : Coupure à q=1 (Signature)")

# 3. PACF - Décroissance
plot_pacf(y_ma1, lags=20, ax=ax[2], title="PACF : Décroissance (Inverse de l'AR)")
plt.show()

##**📝 Analyse de la Signature (MA)**

C'est l'inverse exact de l'AR !

**ACF (Milieu)** : Elle coupe net après le Lag 1. Un choc vieux de 2 jours n'a plus d'effet direct.

**PACF (Droite**) : Elle décroît en ondes ou exponentiellement.

**Règle d'Or** : ***Si l'ACF coupe après le lag $q$, c'est un processus MA($q$)***.

#**4️⃣ Partie III : Le Mixte (ARMA)**
Quand les deux mémoires se mélangent, les signatures visuelles deviennent floues.

In [ ]:
# --- SIMULATION ARMA(1,1) ---
# y_t = 0.7*y_{t-1} + e_t + 0.5*e_{t-1}
ar_params = np.array([1, -0.7])
ma_params = np.array([1, 0.5])

arma_process = ArmaProcess(ar_params, ma_params)
np.random.seed(10)
y_arma = arma_process.generate_sample(nsample=300)

# --- VISUALISATION ---
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].plot(y_arma, color='#8b5cf6')
ax[0].set_title("Série ARMA(1,1)")

plot_acf(y_arma, lags=20, ax=ax[1], title="ACF : Décroissance Mixte")
plot_pacf(y_arma, lags=20, ax=ax[2], title="PACF : Décroissance Mixte")
plt.show()

##**📝 Le Problème de l'ARMA**
Ici, an n'a pas une coupure visuellement nette.

L'ACF ***descend doucement***, la PACF ***aussi***.

Impossible de dire à l'œil nu si c'est (1,1), (2,1) ou (1,2).

**Solution :** On utilise le critère AIC (**Akaike Information Criterion**). On teste plusieurs combinaisons, et **on garde celle qui a l'AIC le plus bas**.

## **Le Juge de Paix : Le Critère AIC (Akaike)**

Puisque l'œil humain ne peut pas départager un ARMA(1,1) d'un ARMA(2,1) sur le graphique, nous utilisons donc ce score mathématique : l'**AIC**.

**La Formule du Juge :**
$$AIC = 2k - 2\ln(\hat{L})$$

Où :
* **$\hat{L}$ (Vraisemblance)** : Mesure à quel point le modèle "colle" aux données (plus c'est haut, mieux c'est).
* **$k$ (Complexité)** : Nombre de paramètres à estimer ($p + q + 1$).
* **$2k$ (Pénalité)** : C'est le "coût" de la complexité. Chaque paramètre ajouté alourdit le score.

> **Règle de Décision (Rasoir d'Ockham)** :
> On cherche le modèle avec l'**AIC le plus faible**.
> Cela signifie qu'on veut un modèle qui explique bien les données ($\ln(L)$ élevé) mais avec un minimum de paramètres ($k$ faible).

### Pourquoi AIC est nécessaire — et pourquoi il n'est pas un oracle

La vraisemblance récompense un modèle capable de rendre les observations vues plausibles. Mais ajouter des paramètres améliore presque toujours l'ajustement in-sample.

AIC impose alors un coût à la complexité :

\[
AIC=-2\log L+2k.
\]

- \(-2\log L\) : coût de mauvais ajustement ;
- \(2k\) : pénalité pour le nombre de paramètres.

On compare les AIC **entre modèles ajustés aux mêmes données et à la même cible**. Le plus petit est préféré dans ce cadre, mais cela ne prouve pas qu'il prévoira mieux le futur. La validation hors échantillon reste indispensable.


##**Pratique (Grid Search AIC)**

In [ ]:
# --- ALGORITHME DE SÉLECTION AUTOMATIQUE (GRID SEARCH) ---

# 1. On définit une grille de recherche
p_values = range(0, 4)  # On teste p = 0, 1, 2, 3
q_values = range(0, 4)  # On teste q = 0, 1, 2, 3
results_list = []

print("🤖 Démarrage du scan des modèles (Grid Search)...")

# 2. On boucle sur toutes les combinaisons
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

for p in p_values:
    for q in q_values:
        if p == 0 and q == 0: continue # On ignore le bruit blanc pur

        try:
            # On entraine le modèle ARIMA(p, 0, q) sur notre série mystère y_arma
            model = ARIMA(y_arma, order=(p, 0, q))
            res = model.fit()

            # On stocke le score AIC
            results_list.append({'p': p, 'q': q, 'AIC': res.aic})

        except:
            continue

# 3. On affiche le classement
df_results = pd.DataFrame(results_list)
df_results = df_results.sort_values(by='AIC', ascending=True).reset_index(drop=True)

print("\n🏆 LE PODIUM DES MEILLEURS MODÈLES :")
print(df_results.head(5))

# 4. Vérification
best_p = df_results.iloc[0]['p']
best_q = df_results.iloc[0]['q']
print(f"\n✅ Le vainqueur est ARMA({best_p},{best_q}).")
print(f"Rappel : La vraie série simulée était un ARMA(1,1).")

if best_p == 1 and best_q == 1:
    print("🎯 VICTOIRE ! L'AIC a retrouvé la vraie structure cachée.")
else:
    print("⚠️ Note : Parfois, sur des petits échantillons, l'AIC peut hésiter avec un modèle proche.")

#**6️⃣ Partie IV : La Grande Synthèse (ARIMA)**

Jusqu'à présent, nous avons travaillé sur des séries stationnaires. Mais **dans la vraie vie, les séries bougent (Tendance, Marche Aléatoire)**.

C'est ici qu'intervient le **"I"** de AR**I**MA (***Integrated***).

L'algorithme complet de **Box-Jenkins** :

**Stationnarisation ($d$)** : On différencie la série jusqu'à ce qu'elle soit plate (cf. Lab 2).

**Identification ($p, q$)** : On regarde l'ACF/PACF de la série différenciée.

**Estimation :** On entraîne le modèle **ARIMA(p,d,q)**.

**Validation :** On vérifie que les résidus sont un bruit blanc.

**Prévision :** On projette vers le futur.

In [ ]:
### 🔬 Simulation d'une "Vraie" Série (Non-Stationnaire)
# Nous allons générer un processus ARIMA(1, 1, 0).
# Cela signifie :
# 1. Elle n'est pas stationnaire (d=1, comme une marche aléatoire).
# 2. Ses variations (la différence) suivent un AR(1) (phi=0.7).

np.random.seed(123)

# 1. On crée la dynamique des variations (AR(1))
# equation : Delta y_t = 0.7 * Delta y_{t-1} + bruit
ar_params = np.array([1, -0.7])
ma_params = np.array([1])
ar1_process = ArmaProcess(ar_params, ma_params)
delta_y = ar1_process.generate_sample(nsample=200)

# 2. On "Intègre" (Somme Cumulée) pour obtenir la série brute (Prix, PIB...)
y_real = np.cumsum(delta_y) + 100  # On commence à 100

# --- VISUALISATION DU PROBLÈME ---
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Série Brute
ax[0].plot(y_real, color='#ef4444')
ax[0].set_title("1. Série Brute (Non-Stationnaire)\nOn ne peut pas modéliser ça directement !")

# Série Différenciée
ax[1].plot(delta_y, color='#10b981')
ax[1].set_title("2. Série Différenciée ($d=1$)\nStationnaire -> Prête pour ACF/PACF")
plt.show()

##**7️⃣ Identification et Modélisation**

Maintenant que nous avons différencié, identifions $p$ et $q$ sur la série verte.

In [ ]:
# --- ÉTAPE 1 : Identification (ACF/PACF sur la différence) ---
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# ACF
plot_acf(delta_y, lags=20, ax=ax[0], title="ACF de la Différence (Décroissance)")

# PACF
plot_pacf(delta_y, lags=20, ax=ax[1], title="PACF de la Différence (Coupure à 1)")
plt.show()

print("🕵️‍♂️ ANALYSE DES SIGNATURES :")
print("- La PACF coupe nettement après le Lag 1.")
print("- L'ACF décroît doucement.")
print("👉 DIAGNOSTIC : C'est un AR(1) sur la différence.")
print("👉 MODÈLE FINAL : ARIMA(p=1, d=1, q=0)")

# --- ÉTAPE 2 : Estimation du Modèle ---
from statsmodels.tsa.arima.model import ARIMA

# Notez bien : on passe la série BRUTE (y_real) et on précise order=(1,1,0)
# Statsmodels va gérer la différenciation tout seul.
model = ARIMA(y_real, order=(1, 1, 0))
model_fit = model.fit()

print("\n--- RÉSULTATS DE L'ESTIMATION ---")
print(model_fit.summary())

##**📝 Commentaire sur les Résultats**
Regardez la table "ar.L1" dans le résumé ci-dessus.

La valeur estimée (coef) est proche de 0.7 (notre valeur simulée).

La P-value (P>|z|) doit être < 0.05 (significatif).

##**8️⃣ Le Grand Final : La Prévision**
C'est le moment de vérité. Le modèle va projeter la dynamique 30 jours dans le futur.

### Avant le graphique de prévision : trois objets

À la date \(t\), pour un horizon \(h\) :

\[
\hat X_{t+h|t}
\]

est la **prévision ponctuelle** construite uniquement avec l'information disponible à \(t\).

L'erreur future est

\[
e_{t+h|t}=X_{t+h}-\hat X_{t+h|t}.
\]

Un intervalle de prévision cherche à couvrir la **future observation**, pas un paramètre inconnu. Il s'élargit généralement avec l'horizon parce que l'incertitude s'accumule.


In [ ]:
# --- 1. PRÉPARATION DES DONNÉES (SPLIT) ---
# On reprend notre série simulée y_real, mais on imagine qu'elle a 230 points
# On coupe en deux :
# - Train (Apprentissage) : Les 200 premiers jours
# - Test (Réalité Future) : Les 30 derniers jours (cachés au modèle)

n_total = 230
train_size = 200

# Régénération propre pour l'exemple
np.random.seed(123)
ar1_diff = ArmaProcess([1, -0.7], [1]).generate_sample(n_total)
y_full = np.cumsum(ar1_diff) + 100

y_train = y_full[:train_size]  # De 0 à 199
y_test = y_full[train_size:]   # De 200 à 229

# --- 2. ENTRAÎNEMENT (Sur le Train uniquement !) ---
model = ARIMA(y_train, order=(1, 1, 0))
model_fit = model.fit()

# --- 3. PRÉVISION (Sur l'horizon du Test) ---
forecast_res = model_fit.get_forecast(steps=len(y_test))
pred_mean = forecast_res.predicted_mean
pred_conf = forecast_res.conf_int()

# --- 4. VISUALISATION (Le Juge de Paix) ---
plt.figure(figsize=(14, 7))

# A. Le Passé (Train)
plt.plot(np.arange(150, 200), y_train[150:], label="Historique (Train)", color='black')

# B. La Réalité Future (Test - Ce qui s'est vraiment passé)
plt.plot(np.arange(200, 230), y_test, label="Réalité (Cachée au modèle)", color='green', linestyle="--", linewidth=2)

# C. La Prévision du Modèle
plt.plot(np.arange(200, 230), pred_mean, label="Prévision ARIMA", color='red', linewidth=2)

# D. L'Intervalle de Confiance
plt.fill_between(
    np.arange(200, 230),
    pred_conf[:, 0],
    pred_conf[:, 1],
    color='pink', alpha=0.3, label="Zone de Confiance (95%)"
)

plt.title("Confrontation : Prévision ARIMA vs Réalité")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

###  Analyse de l'Intervalle de Confiance (Le Cône Rose)

Vous observez une zone rose qui s'élargit avec le temps, formant un "cône" ou une "trompette". Pourquoi ?

1.  **L'Accumulation de l'Incertitude** :

Prédire demain est facile. Prédire dans 30 jours est risqué.
Dans un processus intégré (ARIMA), les erreurs s'accumulent.
Si je me trompe un peu aujourd'hui, cette erreur sert de base pour la prévision de demain, qui ajoute sa propre erreur, etc.

**Mathématiquement :** Pour une marche aléatoire, la variance de l'erreur grandit linéairement avec le temps $t$. L'écart-type (la largeur du cône) grandit comme $\sqrt{t}$.*

2.  **La Signification du 95%** :

Statistiquement, cela signifie que si nous pouvions relancer l'histoire 100 fois à partir d'aujourd'hui, **dans 95 cas sur 100**, la courbe réelle (verte) resterait à l'intérieur de la zone rose.

3.  **Le Verdict Visuel** :
    * Regardez la courbe verte (Réalité).
    * Est-elle restée dans le cône rose ? **OUI**.
    * Le modèle est donc statistiquement valide, même si la ligne rouge ne colle pas parfaitement à la verte. Il a correctement estimé le risque.

##**9️⃣ Bonus Crucial : L'Analyse des Résidus**

Un bon Data Scientist ne s'arrête jamais à la prévision. Il doit vérifier ses poubelles (les résidus).

❗***Si le modèle est bon, il ne doit rester QUE du bruit blanc dans les résidus.***❗

In [ ]:
# Extraction des résidus
residuals = model_fit.resid

# Vérification
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# 1. Trace des résidus
ax[0].plot(residuals[1:], color='purple') # On ignore le premier point (perdu par diff)
ax[0].set_title("Les Résidus (Doit ressembler à du bruit)")

# 2. ACF des résidus
plot_acf(residuals[1:], lags=20, ax=ax[1], title="ACF des Résidus (Doit être vide)")
plt.show()

print("🔍 VERDICT SANITAIRE :")
print("Si l'ACF des résidus ne montre aucune barre sortant de la zone bleue,")
print("alors toute l'information a été capturée. Le modèle est VALIDE ✅.")

## 🔬 Enrichissement Trading Intelligent — vraie évaluation hors échantillon

Un modèle temporel n'est pas évalué comme une régression i.i.d. : on ne mélange jamais passé et futur.

Nous créons une série ARMA synthétique, réservons les 20 % dernières observations comme futur intact, ajustons plusieurs candidats uniquement sur le train, puis comparons leurs prévisions sur le test.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(20261008)
ar = np.array([1, -0.65])
ma = np.array([1, 0.35])
x = ArmaProcess(ar, ma).generate_sample(nsample=700, distrvs=rng.normal)

split = int(0.80 * len(x))
train, test = x[:split], x[split:]

candidates = [(1,0,0), (0,0,1), (1,0,1), (2,0,1)]
rows = []
for order in candidates:
    fit = ARIMA(train, order=order, trend="c").fit()
    pred = fit.forecast(steps=len(test))
    rows.append({
        "order": order,
        "AIC_train": fit.aic,
        "MAE_test": mean_absolute_error(test, pred),
        "RMSE_test": mean_squared_error(test, pred) ** 0.5
    })

results = pd.DataFrame(rows).sort_values("AIC_train")
display(results)

best_order = tuple(results.iloc[0]["order"])
fit = ARIMA(train, order=best_order, trend="c").fit()
pred = fit.forecast(steps=len(test))

import matplotlib.pyplot as plt
plt.figure(figsize=(11,4))
plt.plot(np.arange(split), train, label="Train", lw=.8)
plt.plot(np.arange(split, len(x)), test, label="Futur observé", lw=.9)
plt.plot(np.arange(split, len(x)), pred, label=f"Prévision ARIMA{best_order}", lw=1.2)
plt.axvline(split, color="black", ls="--", lw=1)
plt.legend()
plt.grid(alpha=.2)
plt.title("Ajuster sur le passé, juger sur un futur intact")
plt.show()


### Ce qu'il faut regarder

Si le modèle au plus petit AIC n'est pas exactement celui au plus petit RMSE futur, il n'y a aucune contradiction. Les deux critères ne répondent pas à la même question :

- AIC juge un compromis ajustement/complexité **sur l'échantillon d'estimation** ;
- MAE/RMSE jugent une performance de prévision **sur des observations jamais utilisées pour ajuster le modèle**.

C'est ce décalage qui prépare la validation walk-forward des chapitres suivants.


## ✅ Checklist de compréhension — Lab 2

Vous devez pouvoir expliquer :

1. pourquoi un AR propage un choc sur un horizon potentiellement infini ;
2. pourquoi un MA(q) possède une mémoire directe finie des innovations ;
3. pourquoi ACF et PACF sont des signatures idéalisées, non des règles automatiques ;
4. à quoi sert l'invertibilité d'un MA ;
5. pourquoi AIC pénalise la complexité ;
6. comment ARIMA combine transformation et dynamique ;
7. pourquoi une prévision doit être évaluée sur un futur resté intact.
